# Data Loading

In [1]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [2]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
reviews_df.head()

,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like ga...,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


In [3]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Utilities', 'nan', 'Audio Production', 'Web Publishing', 'Simulation', 'Software Training', 'Early Access', 'Tutorial', 'Free To Play', 'Adventure', 'Animation & Modeling', 'Photo Editing', 'Video Production', 'Short', 'Strategy', 'Violent', 'Massively Multiplayer', 'Racing', 'Education', 'Accounting', 'Sexual Content', 'Episodic', '360 Video', 'Action', 'Design & Illustration', 'Movie', 'Gore', 'Nudity', 'Casual', 'Game Development', 'Indie', 'RPG', 'Documentary', 'Sports'}
34
{'Runner', 'Atmospheric', 'Immersive', 'Hunting', 'Jump Scare', 'World War II', 'Remake', 'Board Game', 'Platformer', 'Souls-like', 'Golf', 'Dog', 'Tactical RPG', 'Old School', 'Underwater', 'Pinball', 'MOBA', 'Linear', 'Bowling', '360 Video', 'Co-op Campaign', 'Party', 'Economy', 'Mythology', 'Narration', 'Fast-Paced', 'Point & Click', 'Based On A Novel', 'Turn-Based Strategy', 'JRPG', 'Sokoban', 'Cinematic', "Shoot 'Em Up", 'Text-Based', 'Asymmetric VR', 'Crime', 'CRPG', 'Mystery Dungeon', 'PvE', 'Surreal', 

In [4]:

narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [5]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


1236465
1101


,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1


## Data Cleaning

In [ ]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

print(filtered_reviews)

         app_id                    app_name  \
46177    104600  Portal 2 - The Final Hours   
46178    104600  Portal 2 - The Final Hours   
46179    104600  Portal 2 - The Final Hours   
46180    104600  Portal 2 - The Final Hours   
46181    104600  Portal 2 - The Final Hours   
...         ...                         ...   
6404109    9880            Champions Online   
6404110    9880            Champions Online   
6404111    9880            Champions Online   
6404112    9880            Champions Online   
6404113    9880            Champions Online   

                                               review_text  review_score  \
46177    First of all this is no game but a virtual int...             1   
46178    The Final Hours is not a game though can't you...             1   
46179    Get this if you are a geek about how games are...             1   
46180    IT'S NOT A GAME | But it's a very interesting ...             1   
46181    Everything you would want to know about Portal

In [8]:
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1
...,...,...,...,...,...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0
6404110,9880,Champions Online,best character creator i've seen | edit: they ...,-1,0
6404111,9880,Champions Online,region banned | why did u show me this game to...,-1,0
6404112,9880,Champions Online,why the,-1,0


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews

# Exploratory Data Analysis

In [13]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices


filtered_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79


### enable gpu

In [2]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.1+cu126
True
NVIDIA GeForce RTX 3060 Laptop GPU


# Sentiment Analysis

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    com_sent.append(doc._.sentiment['compound'])
    pos_sent.append(doc._.sentiment['pos'])
    neg_sent.append(doc._.sentiment['neg'])
    neu_sent.append(doc._.sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()

35000/1236465


KeyboardInterrupt: 

# Topic Modeling

## Auto Topic Modeling with Transofmers
Based on the methodology of Kaggle User Ali Marjani, PhD in the following notebook: https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

In [ ]:
import numpy as np 
import pandas as pd 
import random as rn
import re
import nltk
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

pd.set_option("display.max_rows", 600)
pd.set_option("display.max_columns", 500)
pd.set_option("max_colwidth", 400)

rn.seed(a=42)
p = 0.004  # to randomly select n% of the rows

## General Topic Modeling

In [ ]:
#General topic modeling
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import pyLDAvis, pyLDAvis.gensim_models

In [25]:
recco_texts = []
recco_texts_sentences = []

notrecco_texts = []
notrecco_texts_sentences = []

###

recco = filtered_reviews[filtered_reviews["review_score"]==1]
recco_lemmatized = [review for review in recco["lemmatized_reviews"]]
print(len(recco_lemmatized))

for i in range(len(recco_lemmatized)):
    for sentence in recco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            recco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', recco_lemmatized[i])).split())>5:
        recco_texts.append([word for word in (re.sub(r' \| ',' ', recco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

notrecco = filtered_reviews[filtered_reviews["review_score"]==-1]
notrecco_lemmatized = [review for review in notrecco["lemmatized_reviews"]]
print(len(notrecco_lemmatized))

for i in range(len(notrecco_lemmatized)):
    for sentence in notrecco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            notrecco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', notrecco_lemmatized[i])).split())>5:
        notrecco_texts.append([word for word in (re.sub(r' \| ',' ', notrecco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

1088492
147973


In [ ]:
print("RECCO TEXTS LEN:     " + str(len(recco_texts)))
print("RECCO SENTS LEN:     " + str(len(recco_texts_sentences)))
print("NOTRECCO TEXTS LEN:  " + str(len(notrecco_texts)))
print("NOTRECCO SENTS LEN:  " + str(len(notrecco_texts_sentences)))

1025206
3500155
RECCO TEXTS LEN:     894996
RECCO SENTS LEN:     2914029
NOTRECCO TEXTS LEN:  130210
NOTRECCO SENTS LEN:  586126


In [14]:
dictionary = Dictionary(recco_texts + notrecco_texts) # Create a GenSimdictionary
dictionary.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus = [dictionary.doc2bow(text) for text in (recco_texts + notrecco_texts)] # Create a GenSim corpus
lda = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=7
)

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary,mds='mmds')

pyLDAvis.display(vis)

d:\4th year\1st sem\soc comp\final project\projectvenv\Lib\site-packages\sklearn\manifold\_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
d:\4th year\1st sem\soc comp\final project\projectvenv\Lib\site-packages\sklearn\manifold\_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


In [15]:
dictionary2 = Dictionary(recco_texts_sentences + notrecco_texts_sentences) # Create a GenSimdictionary
dictionary2.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus2 = [dictionary2.doc2bow(text) for text in (recco_texts_sentences + notrecco_texts_sentences)] # Create a GenSim corpus
lda2 = LdaModel(
    corpus=corpus2, 
    id2word=dictionary2, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
vis2 = pyLDAvis.gensim_models.prepare(lda2, corpus2, dictionary2,mds='mmds')

pyLDAvis.display(vis2)

### Topic Modeling Across Recommended / Not Recommended

In [28]:
dictionaryReccoTexts = Dictionary(recco_texts) # Create a GenSimdictionary
dictionaryReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRT = [dictionaryReccoTexts.doc2bow(text) for text in recco_texts] # Create a GenSim corpus
ldaRT = LdaModel(
    corpus=corpusRT, 
    id2word=dictionaryReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRT = pyLDAvis.gensim_models.prepare(ldaRT, corpusRT, dictionaryReccoTexts,mds='mmds')

pyLDAvis.display(visRT)

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\sklearn\manifold\_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\sklearn\manifold\_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


In [ ]:
dictionaryReccoSentences = Dictionary(recco_texts_sentences) # Create a GenSimdictionary
dictionaryReccoSentences.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRS = [dictionaryReccoSentences.doc2bow(text) for text in recco_texts_sentences] # Create a GenSim corpus
ldaRS = LdaModel(
    corpus=corpusRS, 
    id2word=dictionaryReccoSentences, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRS = pyLDAvis.gensim_models.prepare(ldaRS, corpusRS, dictionaryReccoSentences,mds='mmds')

pyLDAvis.display(visRS)

In [29]:
dictionaryNotReccoTexts = Dictionary(notrecco_texts) # Create a GenSimdictionary
dictionaryNotReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusNRT = [dictionaryNotReccoTexts.doc2bow(text) for text in notrecco_texts] # Create a GenSim corpus
ldaNRT = LdaModel(
    corpus=corpusNRT, 
    id2word=dictionaryNotReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visNRT = pyLDAvis.gensim_models.prepare(ldaNRT, corpusNRT, dictionaryNotReccoTexts,mds='mmds')

pyLDAvis.display(visNRT)

c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\sklearn\manifold\_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\moira\Desktop\(4) AY 2026-2027\FirstSemester_SY2627\CSCI161.03_SocialComputing\BongcayaoSalazarTanalgo_SocialComputing\venv\Lib\site-packages\sklearn\manifold\_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
